# 02 — Cleaning and feature engineering

## Goal
Convert the verified raw files into analysis-ready tables without destroying useful business signals. The source has already passed candidate-key and referential-integrity checks in Notebook 01.

**Important principle:** missing delivery timestamps are often expected for cancelled, unavailable, or in-progress orders. We retain them; we do not replace them with invented dates.

In [10]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
for folder in (PROCESSED_DIR, TABLE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

file_map = {
    'customers': 'customers.csv',
    'orders': 'orders.csv',
    'order_items': 'order_items.csv',
    'order_payments': 'order_payments.csv',
    'products': 'products.csv',
    'seller': 'seller.csv'
}
raw = {name: pd.read_csv(RAW_DIR / filename, dtype=str) for name, filename in file_map.items()}
{name: frame.shape for name, frame in raw.items()}

{'customers': (99441, 5),
 'orders': (99441, 8),
 'order_items': (112650, 7),
 'order_payments': (103886, 5),
 'products': (32951, 9),
 'seller': (3095, 4)}

## 1. Baseline quality profile

This is the factual record used to justify each decision below. No rows are changed in this section.

In [11]:
quality_rows = []
for table_name, frame in raw.items():
    for column in frame.columns:
        quality_rows.append({
            'table': table_name,
            'column': column,
            'rows': len(frame),
            'missing_rows': int(frame[column].isna().sum()),
            'missing_pct': round(frame[column].isna().mean() * 100, 2),
            'blank_string_rows': int(frame[column].fillna('').str.strip().eq('').sum()),
            'distinct_non_null': int(frame[column].nunique(dropna=True))
        })

quality_baseline = pd.DataFrame(quality_rows)
quality_baseline.to_csv(TABLE_DIR / '02_missing_value_baseline.csv', index=False)
display(quality_baseline.query('missing_rows > 0 or blank_string_rows > 0').sort_values(['table', 'missing_rows'], ascending=[True, False]))

,table,column,rows,missing_rows,missing_pct,blank_string_rows,distinct_non_null
11,orders,order_delivered_customer_date,99441,2965,2.98,2965,95664
10,orders,order_delivered_carrier_date,99441,1783,1.79,1783,81018
9,orders,order_approved_at,99441,160,0.16,160,90733
26,products,product_category_name,32951,610,1.85,610,73
27,products,product_name_lenght,32951,610,1.85,610,66
28,products,product_description_lenght,32951,610,1.85,610,2960
29,products,product_photos_qty,32951,610,1.85,610,19
30,products,product_weight_g,32951,2,0.01,2,2204
31,products,product_length_cm,32951,2,0.01,2,99
32,products,product_height_cm,32951,2,0.01,2,102


## 2. Parse data types with an audit trail

IDs remain strings. We parse only known timestamp and numeric business fields. `invalid_after_parse` counts non-empty source values that cannot be converted, which must be investigated rather than silently treated as missing.

In [12]:
clean = {name: frame.copy() for name, frame in raw.items()}
date_columns = [
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
numeric_columns = {
    'order_items': ['order_item_id', 'price', 'freight_value'],
    'order_payments': ['payment_sequential', 'payment_installments', 'payment_value'],
    'products': ['product_name_lenght', 'product_description_lenght', 'product_photos_qty',
                 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
}

conversion_log = []
for column in date_columns:
    source = clean['orders'][column]
    parsed = pd.to_datetime(source, errors='coerce')
    conversion_log.append({'table': 'orders', 'column': column, 'target_type': 'datetime64',
                           'non_null_source': int(source.notna().sum()),
                           'invalid_after_parse': int((source.notna() & parsed.isna()).sum())})
    clean['orders'][column] = parsed

for table_name, columns in numeric_columns.items():
    for column in columns:
        source = clean[table_name][column]
        parsed = pd.to_numeric(source, errors='coerce')
        conversion_log.append({'table': table_name, 'column': column, 'target_type': 'numeric',
                               'non_null_source': int(source.notna().sum()),
                               'invalid_after_parse': int((source.notna() & parsed.isna()).sum())})
        clean[table_name][column] = parsed

conversion_log = pd.DataFrame(conversion_log)
conversion_log.to_csv(TABLE_DIR / '02_type_conversion_audit.csv', index=False)
display(conversion_log)

,table,column,target_type,non_null_source,invalid_after_parse
0,orders,order_purchase_timestamp,datetime64,99441,0
1,orders,order_approved_at,datetime64,99281,0
2,orders,order_delivered_carrier_date,datetime64,97658,0
3,orders,order_delivered_customer_date,datetime64,96476,0
4,orders,order_estimated_delivery_date,datetime64,99441,0
5,order_items,order_item_id,numeric,112650,0
6,order_items,price,numeric,112650,0
7,order_items,freight_value,numeric,112650,0
8,order_payments,payment_sequential,numeric,103886,0
9,order_payments,payment_installments,numeric,103886,0


## 3. Validate business rules before creating features

We do not delete violations in this notebook. A negative price, negative freight, impossible installment count, or backwards order timestamp is recorded for review.

In [13]:
orders = clean['orders']
items = clean['order_items']
payments = clean['order_payments']

chronology_pairs = [
    ('order_purchase_timestamp', 'order_approved_at'),
    ('order_approved_at', 'order_delivered_carrier_date'),
    ('order_delivered_carrier_date', 'order_delivered_customer_date')
]
rule_results = [
    {'rule': 'negative_item_price', 'violating_rows': int((items['price'] < 0).sum())},
    {'rule': 'negative_freight', 'violating_rows': int((items['freight_value'] < 0).sum())},
    {'rule': 'non_positive_payment_value', 'violating_rows': int((payments['payment_value'] <= 0).sum())},
    {'rule': 'negative_payment_installments', 'violating_rows': int((payments['payment_installments'] < 0).sum())}
]
for earlier, later in chronology_pairs:
    comparable = orders[earlier].notna() & orders[later].notna()
    rule_results.append({
        'rule': f'{later}_before_{earlier}',
        'violating_rows': int((orders.loc[comparable, later] < orders.loc[comparable, earlier]).sum())
    })

business_rule_report = pd.DataFrame(rule_results)
business_rule_report.to_csv(TABLE_DIR / '02_business_rule_report.csv', index=False)
display(business_rule_report)
display(orders['order_status'].value_counts(dropna=False).rename_axis('order_status').reset_index(name='orders'))

,rule,violating_rows
0,negative_item_price,0
1,negative_freight,0
2,non_positive_payment_value,9
3,negative_payment_installments,0
4,order_approved_at_before_order_purchase_timestamp,0
5,order_delivered_carrier_date_before_order_appr...,1359
6,order_delivered_customer_date_before_order_del...,23


,order_status,orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


## 4. Apply transparent, minimal cleaning decisions

- Keep all valid rows: the previous notebook found no duplicate keys or orphan links.
- Keep lifecycle dates missing where the order did not reach that stage.
- Retain products with missing catalog metadata; label only the missing category as `unknown` so category aggregation remains complete.
- Do not impute physical dimensions or prices. Imputation would invent operational facts.

In [14]:
decision_log = pd.DataFrame([
    {'object': 'all tables', 'decision': 'retain all rows', 'reason': 'No duplicate candidate keys or orphan foreign keys in Notebook 01.'},
    {'object': 'orders lifecycle dates', 'decision': 'retain missing values', 'reason': 'A missing date can legitimately indicate an uncompleted stage.'},
    {'object': 'products.product_category_name', 'decision': "fill missing with 'unknown'", 'reason': 'Preserves sold products in category-level totals while making incompleteness explicit.'},
    {'object': 'product dimensions and catalog fields', 'decision': 'retain missing values', 'reason': 'No defensible source-based value exists for imputation.'}
])
clean['products']['product_category_name'] = clean['products']['product_category_name'].fillna('unknown')
decision_log.to_csv(TABLE_DIR / '02_cleaning_decision_log.csv', index=False)
display(decision_log)

,object,decision,reason
0,all tables,retain all rows,No duplicate candidate keys or orphan foreign ...
1,orders lifecycle dates,retain missing values,A missing date can legitimately indicate an un...
2,products.product_category_name,fill missing with 'unknown',Preserves sold products in category-level tota...
3,product dimensions and catalog fields,retain missing values,No defensible source-based value exists for im...


## 5. Create analysis features at the correct grain

We create two separate facts:

- **order_fact**: exactly one row per order; payments and items are aggregated to order level first.
- **item_fact**: exactly one row per order item; used for product and seller analysis.

This avoids the classic many-to-many error of joining raw payments to raw items, which would multiply revenue.

In [15]:
orders = clean['orders'].copy()
orders['purchase_year'] = orders['order_purchase_timestamp'].dt.year
orders['purchase_month'] = orders['order_purchase_timestamp'].dt.to_period('M').astype('string')
orders['purchase_weekday'] = orders['order_purchase_timestamp'].dt.day_name()
orders['approval_hours'] = (orders['order_approved_at'] - orders['order_purchase_timestamp']).dt.total_seconds() / 3600
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.total_seconds() / 86400
orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.total_seconds() / 86400
orders['is_delivered'] = orders['order_status'].eq('delivered')
orders['is_late'] = orders['delivery_delay_days'].gt(0)

items = clean['order_items'].copy()
items['item_total_value'] = items['price'] + items['freight_value']

products = clean['products'].copy()
products['product_volume_cm3'] = products['product_length_cm'] * products['product_height_cm'] * products['product_width_cm']

item_summary = (items.groupby('order_id', as_index=False)
    .agg(item_count=('order_item_id', 'size'),
         merchandise_value=('price', 'sum'),
         freight_value=('freight_value', 'sum'),
         order_item_total=('item_total_value', 'sum'),
         seller_count=('seller_id', 'nunique'),
         product_count=('product_id', 'nunique')))

payment_summary = (clean['order_payments'].groupby('order_id', as_index=False)
    .agg(payment_value=('payment_value', 'sum'),
         payment_records=('payment_sequential', 'size'),
         payment_methods=('payment_type', 'nunique'),
         max_installments=('payment_installments', 'max')))

order_fact = (orders
    .merge(item_summary, on='order_id', how='left', validate='one_to_one')
    .merge(payment_summary, on='order_id', how='left', validate='one_to_one'))
order_fact['payment_minus_item_total'] = order_fact['payment_value'] - order_fact['order_item_total']

item_fact = (items
    .merge(orders[['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'is_delivered', 'delivery_days', 'delivery_delay_days']], on='order_id', how='left', validate='many_to_one')
    .merge(products, on='product_id', how='left', validate='many_to_one')
    .merge(clean['seller'], on='seller_id', how='left', validate='many_to_one', suffixes=('', '_seller')))

print('order_fact grain:', order_fact['order_id'].nunique(), 'unique orders /', len(order_fact), 'rows')
print('item_fact grain:', item_fact[['order_id', 'order_item_id']].drop_duplicates().shape[0], 'unique order-items /', len(item_fact), 'rows')

order_fact grain: 99441 unique orders / 99441 rows
item_fact grain: 112650 unique order-items / 112650 rows


In [16]:
# Final reconciliation: investigate material differences; do not assume payment and item totals must always match exactly.
reconciliation = (order_fact[['order_id', 'order_status', 'order_item_total', 'payment_value', 'payment_minus_item_total']]
    .assign(abs_difference=lambda df: df['payment_minus_item_total'].abs())
    .sort_values('abs_difference', ascending=False))
reconciliation.head(10).to_csv(TABLE_DIR / '02_largest_payment_item_differences.csv', index=False)
display(reconciliation.head(10))

# Save reusable outputs for the analysis notebooks.
for name, frame in {**clean, 'order_fact': order_fact, 'item_fact': item_fact}.items():
    frame.to_csv(PROCESSED_DIR / f'{name}_clean.csv', index=False)

print(f'Saved cleaned and analytical CSVs to: {PROCESSED_DIR}')

,order_id,order_status,order_item_total,payment_value,payment_minus_item_total,abs_difference
11791,ce6d150fb29ada17d2082f4847107665,delivered,1403.66,1586.47,182.81,182.81
48686,6e5fe7366a2e1bfbf3257dba0af1267f,delivered,287.91,406.92,119.01,119.01
70865,70b742795bc441e94a44a084b6d9ce7a,delivered,466.93,578.82,111.89,111.89
33150,996c7e73600ad3723e8627ab7bef81e4,delivered,587.90,664.43,76.53,76.53
52985,70b7e94ea46d3e8b5bc12a50186edaf0,delivered,213.15,274.84,61.69,61.69
85434,bc2c82b0ef78d2252b6176d1972db7c9,delivered,242.01,303.02,61.01,61.01
94304,af9ffff2ce6b3defd34fd4c78857a379,delivered,413.17,466.97,53.80,53.80
31661,262118ce178bb3e4590a3adcf6d62e6b,delivered,177.74,126.12,-51.62,51.62
68811,bfdb5bbb06458d600a33d61f5f287472,delivered,348.93,394.36,45.43,45.43
8548,8d9c0dc8d5a2ce804f6b925d8f8e6c1d,delivered,254.45,293.89,39.44,39.44


Saved cleaned and analytical CSVs to: C:\Users\Muhamed\Documents\Codex\2026-08-11\referenced-chatgpt-conversation-this-is-an\data\processed


## 6. Quarantine flags and anomaly diagnostics

We preserve anomalies in the data, but exclude only invalid timelines from duration averages. The financial investigation below describes payment/item differences before deciding which metric answers each business question.

In [17]:
# A valid delivery duration requires a completed delivery after (or at) purchase.
order_fact['valid_delivery_timeline'] = (
    order_fact['order_delivered_customer_date'].notna()
    & order_fact['order_purchase_timestamp'].notna()
    & order_fact['order_delivered_customer_date'].ge(order_fact['order_purchase_timestamp'])
)
order_fact['delivery_days_valid'] = order_fact['delivery_days'].where(order_fact['valid_delivery_timeline'])

timeline_summary = (order_fact.groupby('order_status', dropna=False)
    .agg(orders=('order_id', 'size'),
         valid_delivery_timelines=('valid_delivery_timeline', 'sum'),
         avg_delivery_days_valid=('delivery_days_valid', 'mean'))
    .reset_index())
timeline_summary.to_csv(TABLE_DIR / '02_delivery_timeline_quality.csv', index=False)
display(timeline_summary)

# A zero-value payment may be a valid voucher/administrative record, so describe it before excluding it.
zero_value_payments = clean['order_payments'].query('payment_value == 0').copy()
zero_payment_summary = (zero_value_payments.groupby('payment_type', dropna=False)
    .agg(records=('order_id', 'size'), orders=('order_id', 'nunique'),
         installments=('payment_installments', 'unique'))
    .reset_index())
zero_payment_summary.to_csv(TABLE_DIR / '02_zero_value_payment_review.csv', index=False)
display(zero_payment_summary)

,order_status,orders,valid_delivery_timelines,avg_delivery_days_valid
0,approved,2,0,NaN
1,canceled,625,6,20.360006
2,created,5,0,NaN
3,delivered,96478,96470,12.558217
4,invoiced,314,0,NaN
5,processing,301,0,NaN
6,shipped,1107,0,NaN
7,unavailable,609,0,NaN


,payment_type,records,orders,installments
0,not_defined,3,3,[1]
1,voucher,6,5,[1]


In [18]:
# Summarise payment structure for orders whose payment total differs materially from item + freight total.
payment_detail = (clean['order_payments'].groupby('order_id', as_index=False)
    .agg(payment_type_list=('payment_type', lambda s: ' | '.join(sorted(s.dropna().unique()))),
         payment_record_count=('payment_sequential', 'size')))

material_difference = (order_fact.assign(abs_difference=lambda df: df['payment_minus_item_total'].abs())
    .query('abs_difference > 0.01')
    .merge(payment_detail, on='order_id', how='left', validate='one_to_one'))
difference_summary = (material_difference.groupby(['payment_type_list', 'payment_record_count'], dropna=False)
    .agg(orders=('order_id', 'size'),
         avg_abs_difference=('abs_difference', 'mean'),
         max_abs_difference=('abs_difference', 'max'))
    .reset_index()
    .sort_values(['orders', 'avg_abs_difference'], ascending=[False, False]))
difference_summary.to_csv(TABLE_DIR / '02_payment_item_difference_summary.csv', index=False)
display(difference_summary.head(20))

# Persist the two valid-duration fields for later delivery analysis.
order_fact.to_csv(PROCESSED_DIR / 'order_fact.csv', index=False)

,payment_type_list,payment_record_count,orders,avg_abs_difference,max_abs_difference
1,credit_card,1,319,9.936928,182.81
0,boleto,1,29,0.016207,0.04
2,credit_card,2,11,2.740000,9.30
4,debit_card,1,9,5.774444,16.50
3,credit_card | voucher,2,9,1.944444,9.26
5,voucher,1,1,0.010000,0.01
